In [44]:
from langgraph.graph import START,END,StateGraph
from typing import TypedDict,Annotated
from langchain_core.messages import BaseMessage,HumanMessage,AIMessage,SystemMessage
from langchain_ollama import ChatOllama
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3
from langchain_core.messages import trim_messages,RemoveMessage
from langchain_core.messages.utils import count_tokens_approximately



In [45]:
llm = ChatOllama(model='qwen2.5:0.5b-instruct',temperature=0.2,max_token=100)

In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]


In [ ]:
def chat_node(state:ChatState):
    messages = trim_messages(
        messages=state['messages'],
        strategy='last',
        max_tokens=150,
        token_counter=count_tokens_approximately
    )
    res = llm.invoke(messages)
    return {'messages':[res]}

def delete_old_messages(state:ChatState):
    messages = state['messages']
    if len(messages) > 10: 
        to_remove = messages[:5] # delete first 5 messages
        return {'messages':[RemoveMessage(id=m.id) for m in to_remove]}
    return {}

In [ ]:
graph = StateGraph(ChatState)

graph.add_node('chat_node',chat_node)
graph.add_node('cleanup',delete_old_messages)

graph.add_edge(START,'chat_node')
graph.add_edge('chat_node','cleanup')
graph.add_edge('cleanup',END)

conn = sqlite3.connect('database/store.db',check_same_thread=False)
chatbot = graph.compile(checkpointer=SqliteSaver(conn=conn))


In [38]:
thread_id = '2'
while True:
    user_message = input("Ask anything: ")
    print("Human : ",user_message)

    if user_message.strip().lower() in ['exit','bye','quit']:
        break

    config = {'configurable':{'thread_id':thread_id}}
    res = chatbot.invoke({'messages':[HumanMessage(content=user_message)]},config=config)

    print('AI : ',res['messages'][-1].content)

Human :  exit


In [49]:
config = {'configurable':{'thread_id':'thread1'}}
chatbot.invoke({'messages':[HumanMessage(content="my name is girish")]},config=config)
chatbot.invoke({'messages':[HumanMessage(content="what is your name")]},config=config)
chatbot.invoke({'messages':[HumanMessage(content="how are you doing")]},config=config)

{'messages': [HumanMessage(content='what is your name', additional_kwargs={}, response_metadata={}, id='16164339-396a-4047-96cd-c78c220e4f42'),
  AIMessage(content='I am Qwen, a large language model created by Alibaba Cloud. My name is Qwen.', additional_kwargs={}, response_metadata={'model': 'qwen2.5:0.5b-instruct', 'created_at': '2026-09-21T17:11:26.729783937Z', 'done': True, 'done_reason': 'stop', 'total_duration': 647483940, 'load_duration': 189666411, 'prompt_eval_count': 83, 'prompt_eval_duration': 92990000, 'eval_count': 21, 'eval_duration': 362551000, 'logprobs': None, 'model_name': 'qwen2.5:0.5b-instruct', 'model_provider': 'ollama'}, id='lc_run--01a0c4f3-7a81-79a3-9958-68a7f6dd7d14-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 83, 'output_tokens': 21, 'total_tokens': 104}),
  HumanMessage(content='how are you doing', additional_kwargs={}, response_metadata={}, id='8397bc24-1cec-40c2-8fe8-923ef407d0fb'),
  AIMessage(content="As an AI language model,

In [50]:
chatbot.invoke({'messages':[HumanMessage(content="what is my name")]},config=config)

{'messages': [HumanMessage(content='how are you doing', additional_kwargs={}, response_metadata={}, id='8397bc24-1cec-40c2-8fe8-923ef407d0fb'),
  AIMessage(content="As an AI language model, I don't have feelings or emotions like humans do. However, I'm here to help answer any questions you might have! How can I assist you today?", additional_kwargs={}, response_metadata={'model': 'qwen2.5:0.5b-instruct', 'created_at': '2026-09-21T17:11:27.701714633Z', 'done': True, 'done_reason': 'stop', 'total_duration': 955813715, 'load_duration': 193282555, 'prompt_eval_count': 117, 'prompt_eval_duration': 67263000, 'eval_count': 39, 'eval_duration': 692917000, 'logprobs': None, 'model_name': 'qwen2.5:0.5b-instruct', 'model_provider': 'ollama'}, id='lc_run--01a0c4f3-7d19-70c2-85e0-a3f4e6897941-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 117, 'output_tokens': 39, 'total_tokens': 156}),
  HumanMessage(content='what is my name', additional_kwargs={}, response_metadata={}, 